# Colab 4: IsoFLOP Scaling Laws — ARM vs MDM vs π-Learners
## Reproducing Figure 2 (Left) and Figure 2 (Top-Right)
### "Train for the Worst, Plan for the Best" (Kim et al., ICML 2025, arXiv:2502.06768v3)

This notebook reproduces:
- **Figure 2 (Left)**: IsoFLOP scaling law curves for ARM, MDM, and 9 π-learners on SlimPajama
- **Figure 2 (Top-Right)**: Validation loss comparison table for π=id vs π~Closer(S_L) vs π~Unif(S_L)

**Key design choices from the paper:**
- All models use **learned positional embeddings** (Section 3.2)
- Tokenizer: GPT-2 (vocab 50257), with mask token added as token 0 (total vocab 50258)
- Sequence length L=2048
- MDM: bidirectional attention, no time embedding
- ARM/π-learners: causal (left-to-right) attention

**Runtime**: A100 GPU recommended. Training all models takes ~24–48 hours depending on GPU.

---
## Cell 1: Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

dirs = [
    f'{BASE_DIR}/results/scaling_laws',
    f'{BASE_DIR}/figures',
    f'{BASE_DIR}/models/scaling_laws',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print('Directory structure ready at:', BASE_DIR)

In [ ]:
!pip install -q torch transformers numpy matplotlib

---
## Cell 2: Imports & Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import json
import math
import time
import os
import copy
import matplotlib.pyplot as plt
import matplotlib
from collections import defaultdict

BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'
RESULTS_DIR = f'{BASE_DIR}/results/scaling_laws'
FIGURES_DIR = f'{BASE_DIR}/figures'
MODELS_DIR = f'{BASE_DIR}/models/scaling_laws'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---
## Cell 3: Model Configuration Lookup Table

Model sizes (non-embedding parameters): 10M, 30M, 60M, 100M, 170M, 300M.
The non-embedding parameter count for a transformer is approximately:
  `N = n_layers * (12 * d_model^2 + 13 * d_model) + d_model * vocab_size` (output head)

We define configs excluding the embedding/output head parameters, since
IsoFLOP analysis counts non-embedding params.

In [ ]:
# Vocab: GPT-2 (50257) + 1 mask token = 50258
VOCAB_SIZE = 50258
SEQ_LEN = 2048

# Model configs: (d_model, n_heads, n_layers)
# Chosen so non-embedding param count ~ target
MODEL_CONFIGS = {
    '10M':  {'d_model': 384,  'n_heads': 6,  'n_layers': 6},
    '30M':  {'d_model': 512,  'n_heads': 8,  'n_layers': 8},
    '60M':  {'d_model': 640,  'n_heads': 10, 'n_layers': 10},
    '100M': {'d_model': 768,  'n_heads': 12, 'n_layers': 12},
    '170M': {'d_model': 1024, 'n_heads': 16, 'n_layers': 12},
    '300M': {'d_model': 1280, 'n_heads': 16, 'n_layers': 16},
}

def count_non_embedding_params(model):
    """Count parameters excluding token and positional embeddings."""
    total = sum(p.numel() for p in model.parameters())
    emb_params = 0
    if hasattr(model, 'token_embedding'):
        emb_params += model.token_embedding.weight.numel()
    if hasattr(model, 'pos_embedding'):
        emb_params += model.pos_embedding.numel()
    return total - emb_params

# Verify configs
print('Model config verification (approximate non-embedding params):')
for name, cfg in MODEL_CONFIGS.items():
    d = cfg['d_model']
    L = cfg['n_layers']
    # Each layer: 4*d*d (QKV+O in attn) * 3 projections = 4*d^2 for Q,K,V each + 1 output = 4*d^2
    # Actually: attn = 4*d^2, FFN = 8*d^2 (up + down, 4x expansion), total per layer = 12*d^2
    # Plus layer norms, biases ~= 13*d per layer
    # Plus output projection d*vocab
    n_nonemb = L * (12 * d**2 + 13 * d) + d * VOCAB_SIZE
    print(f'  {name}: d={d}, heads={cfg["n_heads"]}, layers={L} -> ~{n_nonemb/1e6:.1f}M params')

---
## Cell 4: Transformer Building Blocks

Shared components: multi-head attention, feed-forward network, transformer block.
All use pre-norm (LayerNorm before attention/FFN) as is standard.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention with optional causal masking."""

    def __init__(self, d_model, n_heads, causal=False):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.causal = causal

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x)  # (B, L, 3*D)
        qkv = qkv.reshape(B, L, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, L, d_head)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scale = self.d_head ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, H, L, L)

        if self.causal:
            causal_mask = torch.triu(
                torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1
            )
            attn = attn.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)  # (B, H, L, d_head)
        out = out.permute(0, 2, 1, 3).reshape(B, L, D)
        return self.out_proj(out)


class FeedForward(nn.Module):
    """Position-wise FFN with 4x expansion."""

    def __init__(self, d_model):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model, bias=False)
        self.fc2 = nn.Linear(4 * d_model, d_model, bias=False)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class TransformerBlock(nn.Module):
    """Pre-norm transformer block."""

    def __init__(self, d_model, n_heads, causal=False):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, causal=causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


print('Transformer building blocks defined.')

---
## Cell 5: ARM Transformer (Causal, Learned Positional Embeddings)

Standard autoregressive model with causal masking.
Token 0 is reserved as the mask token; actual GPT-2 tokens are shifted by +1.
Uses learned positional embeddings (NOT RoPE), per Section 3.2.

In [ ]:
class ARMTransformer(nn.Module):
    """
    Autoregressive (causal) transformer for language modeling.
    Also used for pi-learners (permuted AR).
    
    Architecture:
    - Learned positional embeddings (Section 3.2)
    - Causal (left-to-right) attention mask
    - Pre-norm transformer blocks
    - Vocab: 50258 (GPT-2 50257 + 1 mask token at index 0)
    """

    def __init__(self, d_model, n_heads, n_layers, vocab_size=VOCAB_SIZE, max_len=SEQ_LEN):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size

        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos_embedding, std=0.02)

        # Transformer blocks (causal)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, causal=True)
            for _ in range(n_layers)
        ])

        # Output head
        self.ln_final = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: output projection shares weights with token embedding
        self.output_proj.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        """
        Args:
            x: (B, L) token indices
        Returns:
            logits: (B, L, vocab_size)
        """
        B, L = x.shape
        h = self.token_embedding(x) + self.pos_embedding[:, :L, :]
        for block in self.blocks:
            h = block(h)
        h = self.ln_final(h)
        logits = self.output_proj(h)
        return logits


print('ARMTransformer defined.')

---
## Cell 6: MDM Transformer (Bidirectional, Learned Positional Embeddings)

Masked Diffusion Model: bidirectional attention, no causal mask.
No time/noise-level embedding (Section 3.2: MDM does not use time embedding).
Predicts clean tokens at all masked positions simultaneously.

In [ ]:
class MDMTransformer(nn.Module):
    """
    Masked Diffusion Model transformer.
    
    Architecture:
    - Learned positional embeddings (Section 3.2)
    - Bidirectional (full) attention
    - No time/noise embedding
    - Pre-norm transformer blocks
    """

    def __init__(self, d_model, n_heads, n_layers, vocab_size=VOCAB_SIZE, max_len=SEQ_LEN):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.mask_token_id = 0  # Token 0 = [MASK]

        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos_embedding, std=0.02)

        # Transformer blocks (bidirectional)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, causal=False)
            for _ in range(n_layers)
        ])

        # Output head
        self.ln_final = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying
        self.output_proj.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        """
        Args:
            x: (B, L) token indices (with some positions masked to token 0)
        Returns:
            logits: (B, L, vocab_size) predictions at all positions
        """
        B, L = x.shape
        h = self.token_embedding(x) + self.pos_embedding[:, :L, :]
        for block in self.blocks:
            h = block(h)
        h = self.ln_final(h)
        logits = self.output_proj(h)
        return logits


print('MDMTransformer defined.')

---
## Cell 7: Permutation Sampling Functions (Appendix C.1)

Three distributions over permutations of {0, 1, ..., L-1}:
- **identity**: π = id (equivalent to standard AR)
- **closer**: start from id, apply L/10 random transposition swaps
- **much_closer**: start from id, apply sqrt(L) random transposition swaps
- **uniform**: π ~ Uniform(S_L), fully random permutation

In [ ]:
def sample_identity_permutation(L):
    """Return the identity permutation (= standard AR order)."""
    return np.arange(L)


def sample_closer_permutation(L, rng=None):
    """
    Start from identity, apply L/10 random transposition swaps.
    This produces a permutation 'close' to identity (Appendix C.1).
    """
    if rng is None:
        rng = np.random.default_rng()
    perm = np.arange(L)
    n_swaps = L // 10
    for _ in range(n_swaps):
        i, j = rng.integers(0, L, size=2)
        perm[i], perm[j] = perm[j], perm[i]
    return perm


def sample_much_closer_permutation(L, rng=None):
    """
    Start from identity, apply sqrt(L) random transposition swaps.
    This produces a permutation 'much closer' to identity (Appendix C.1).
    """
    if rng is None:
        rng = np.random.default_rng()
    perm = np.arange(L)
    n_swaps = int(math.sqrt(L))
    for _ in range(n_swaps):
        i, j = rng.integers(0, L, size=2)
        perm[i], perm[j] = perm[j], perm[i]
    return perm


def sample_uniform_permutation(L, rng=None):
    """Sample a uniformly random permutation."""
    if rng is None:
        rng = np.random.default_rng()
    return rng.permutation(L)


# Verification
L_test = 2048
rng = np.random.default_rng(42)
print(f'Closer: {L_test // 10} swaps = {L_test // 10}')
print(f'Much closer: sqrt({L_test}) swaps = {int(math.sqrt(L_test))}')

# Check that closer is indeed closer to identity than uniform
# Kendall tau distance: number of pairwise disagreements
def kendall_tau_distance(perm):
    """Approximate Kendall tau distance from identity."""
    n = len(perm)
    count = 0
    # Sample-based estimate for speed
    n_samples = min(100000, n * (n - 1) // 2)
    for _ in range(n_samples):
        i, j = sorted(rng.integers(0, n, size=2))
        if i != j and perm[i] > perm[j]:
            count += 1
    return count / n_samples

for name, fn in [('identity', sample_identity_permutation),
                  ('much_closer', lambda L: sample_much_closer_permutation(L, rng)),
                  ('closer', lambda L: sample_closer_permutation(L, rng)),
                  ('uniform', lambda L: sample_uniform_permutation(L, rng))]:
    if name == 'identity':
        p = fn(L_test)
    else:
        p = fn(L_test)
    kt = kendall_tau_distance(p)
    print(f'  {name}: Kendall-tau ~ {kt:.4f} (0 = identity, 0.5 = random)')

---
## Cell 8: Data Loading

Load pre-tokenized SlimPajama sequences from Colab 1.
Sequences are GPT-2 tokenized at length L=2048.
We shift all tokens by +1 to reserve token 0 as the mask token.

In [ ]:
# Load tokenized data
train_path = f'{BASE_DIR}/data/slimpajama/train_sequences.npy'
val_path = f'{BASE_DIR}/data/slimpajama/val_sequences.npy'

print('Loading training sequences...')
train_data = np.load(train_path).astype(np.int64)
print(f'  Train: {train_data.shape} (sequences x length)')

print('Loading validation sequences...')
val_data = np.load(val_path).astype(np.int64)
print(f'  Val: {val_data.shape}')

# Shift tokens by +1 to reserve 0 for mask token
train_data = train_data + 1
val_data = val_data + 1

print(f'Token range after shift: [{train_data.min()}, {train_data.max()}]')
assert train_data.min() >= 1, 'Token 0 should be reserved for mask'
assert train_data.max() < VOCAB_SIZE, f'Max token {train_data.max()} >= vocab size {VOCAB_SIZE}'

# Total tokens available
n_train_tokens = train_data.shape[0] * train_data.shape[1]
print(f'Total training tokens: {n_train_tokens:,.0f} ({n_train_tokens/1e9:.2f}B)')

---
## Cell 9: Training Utilities

Core functions for:
- ARM/π-learner training (next-token prediction under permuted order)
- MDM training (masked diffusion objective)
- Cosine learning rate schedule
- Validation loss evaluation

In [ ]:
def get_cosine_lr(step, total_steps, max_lr=4e-4, min_lr=4e-5, warmup_steps=500):
    """Cosine schedule with linear warmup."""
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def apply_permutation_to_sequence(x, perm):
    """
    Reorder sequence x according to permutation perm.
    For pi-learner training: tokens are rearranged so that
    position i contains the token that should be predicted i-th
    in the permuted order.
    
    Args:
        x: (B, L) original token sequences
        perm: (L,) permutation array
    Returns:
        x_perm: (B, L) permuted sequences
    """
    return x[:, perm]


def compute_arm_loss(model, x, perm=None):
    """
    Compute autoregressive loss under a given permutation order.
    
    For standard AR (perm=None or perm=identity):
        loss = -sum_t log p(x_{t+1} | x_{1:t})
    
    For pi-learner (perm given):
        Reorder x by perm, then do standard AR on reordered sequence.
    """
    if perm is not None:
        x = apply_permutation_to_sequence(x, perm)

    logits = model(x)  # (B, L, V)
    # Predict next token: logits[:, t, :] predicts x[:, t+1]
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = x[:, 1:].contiguous()
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        reduction='mean'
    )
    return loss


def compute_mdm_loss(model, x):
    """
    Compute masked diffusion model loss.
    
    MDM training (Sahoo et al. 2024, Shi et al. 2024):
    1. Sample masking rate t ~ Uniform(0, 1)
    2. Mask each position independently with probability t
    3. Predict original tokens at masked positions
    4. Weight loss by 1/t (importance weighting for ELBO)
    """
    B, L = x.shape

    # Sample masking rate
    t = torch.rand(B, 1, device=x.device)  # (B, 1)

    # Create mask: True where position is masked
    mask = torch.rand(B, L, device=x.device) < t  # (B, L)

    # Ensure at least one position is masked per sequence
    # (for numerical stability)
    no_mask = ~mask.any(dim=1)  # (B,)
    if no_mask.any():
        random_pos = torch.randint(0, L, (no_mask.sum(),), device=x.device)
        mask[no_mask, random_pos] = True

    # Apply mask: replace masked positions with mask token (0)
    x_masked = x.clone()
    x_masked[mask] = 0  # mask token id = 0

    # Forward pass
    logits = model(x_masked)  # (B, L, V)

    # Loss only at masked positions, weighted by 1/t
    # Per-token cross-entropy
    per_token_loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        x.view(-1),
        reduction='none'
    ).view(B, L)  # (B, L)

    # Mask and weight
    masked_loss = per_token_loss * mask.float()  # zero out unmasked
    # Weight by 1/t for each sequence (ELBO importance weight)
    t_clamped = t.clamp(min=1e-4)  # avoid division by zero
    weighted_loss = masked_loss / t_clamped  # (B, L)

    # Average over masked positions and batch
    n_masked = mask.float().sum(dim=1, keepdim=True).clamp(min=1)
    loss = (weighted_loss.sum(dim=1, keepdim=True) / n_masked).mean()

    return loss


@torch.no_grad()
def evaluate_arm_loss(model, val_data_tensor, batch_size=64, perm=None):
    """Evaluate ARM/pi-learner validation loss."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for i in range(0, len(val_data_tensor), batch_size):
        x = val_data_tensor[i:i+batch_size].to(device)
        loss = compute_arm_loss(model, x, perm=perm)
        total_loss += loss.item()
        n_batches += 1
    model.train()
    return total_loss / n_batches


@torch.no_grad()
def evaluate_mdm_loss(model, val_data_tensor, batch_size=64, n_eval_samples=10):
    """
    Evaluate MDM validation loss.
    Average over multiple random masking samples for stable estimate.
    """
    model.eval()
    total_loss = 0.0
    n_total = 0
    for _ in range(n_eval_samples):
        for i in range(0, len(val_data_tensor), batch_size):
            x = val_data_tensor[i:i+batch_size].to(device)
            loss = compute_mdm_loss(model, x)
            total_loss += loss.item()
            n_total += 1
    model.train()
    return total_loss / n_total


print('Training utilities defined.')

---
## Cell 10: IsoFLOP Training Pipeline

For each FLOPs budget C, we:
1. Select multiple model sizes from our config table
2. Compute training tokens = C / (6 * N_non_emb_params) (Hoffmann et al. 2022)
3. Train each model for the computed number of tokens
4. Record the best validation loss across model sizes

This traces out the IsoFLOP-optimal frontier.

In [ ]:
# IsoFLOP budgets: ~5-7 values spanning 10^18 to 10^21
FLOPS_BUDGETS = [1e18, 3e18, 1e19, 3e19, 1e20, 3e20, 1e21]

# Model sizes to try for each budget
# For small budgets, only small models make sense (enough tokens to train)
# For large budgets, we can afford larger models
FLOPS_TO_MODEL_SIZES = {
    1e18:  ['10M'],
    3e18:  ['10M', '30M'],
    1e19:  ['10M', '30M', '60M'],
    3e19:  ['30M', '60M', '100M'],
    1e20:  ['60M', '100M', '170M'],
    3e20:  ['100M', '170M', '300M'],
    1e21:  ['170M', '300M'],
}

BATCH_SIZE = 64  # sequences per batch
TOKENS_PER_BATCH = BATCH_SIZE * SEQ_LEN  # tokens per batch

print(f'Tokens per batch: {TOKENS_PER_BATCH:,}')
print(f'\nFLOPs budgets and candidate model sizes:')
for c, sizes in FLOPS_TO_MODEL_SIZES.items():
    print(f'  C = {c:.0e}: {sizes}')

In [ ]:
def train_single_model(model_type, model_config_name, total_training_tokens,
                        train_data_np, val_data_tensor, perm=None, perm_name=None,
                        batch_size=BATCH_SIZE, max_lr=4e-4, min_lr=4e-5,
                        eval_every_tokens=50_000_000, save_dir=None):
    """
    Train a single model for a specified number of tokens.
    
    Args:
        model_type: 'arm', 'mdm', or 'pi_learner'
        model_config_name: key into MODEL_CONFIGS (e.g., '100M')
        total_training_tokens: total tokens to train on
        train_data_np: numpy array of training sequences
        val_data_tensor: torch tensor of validation sequences
        perm: permutation array (for pi-learners), or None
        perm_name: name string for logging
        batch_size: batch size in sequences
        max_lr: maximum learning rate
        min_lr: minimum learning rate
        eval_every_tokens: evaluate every N tokens
        save_dir: directory to save checkpoints
    
    Returns:
        dict with best_val_loss, final_val_loss, training_log
    """
    cfg = MODEL_CONFIGS[model_config_name]
    d_model = cfg['d_model']
    n_heads = cfg['n_heads']
    n_layers = cfg['n_layers']

    # Create model
    if model_type == 'mdm':
        model = MDMTransformer(d_model, n_heads, n_layers).to(device)
    else:
        model = ARMTransformer(d_model, n_heads, n_layers).to(device)

    n_params = count_non_embedding_params(model)
    total_params = sum(p.numel() for p in model.parameters())

    # Compute training steps
    tokens_per_batch = batch_size * SEQ_LEN
    total_steps = int(total_training_tokens / tokens_per_batch)
    eval_every_steps = max(1, int(eval_every_tokens / tokens_per_batch))

    name = f'{model_type}'
    if perm_name:
        name += f'_{perm_name}'
    name += f'_{model_config_name}'

    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'  Non-embedding params: {n_params:,} ({n_params/1e6:.1f}M)')
    print(f'  Total params: {total_params:,}')
    print(f'  Training tokens: {total_training_tokens:,.0f}')
    print(f'  Training steps: {total_steps:,}')
    print(f'  Eval every: {eval_every_steps} steps ({eval_every_tokens/1e6:.0f}M tokens)')
    print(f'{"="*60}')

    if total_steps < 10:
        print(f'  WARNING: Only {total_steps} steps. Skipping (too few).')
        return {'best_val_loss': float('inf'), 'final_val_loss': float('inf'),
                'n_params': n_params, 'training_log': [], 'skipped': True}

    # Optimizer: AdamW
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=max_lr,
        betas=(0.9, 0.95),
        weight_decay=0.1
    )

    # Training loop
    n_train_seqs = len(train_data_np)
    best_val_loss = float('inf')
    training_log = []
    running_loss = 0.0
    log_interval = max(1, total_steps // 20)

    model.train()
    t_start = time.time()

    for step in range(total_steps):
        # Learning rate schedule
        lr = get_cosine_lr(step, total_steps, max_lr, min_lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        # Sample batch
        idx = np.random.randint(0, n_train_seqs, size=batch_size)
        x = torch.from_numpy(train_data_np[idx]).to(device)

        # Forward + backward
        if model_type == 'mdm':
            loss = compute_mdm_loss(model, x)
        else:
            loss = compute_arm_loss(model, x, perm=perm)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()

        # Logging
        if (step + 1) % log_interval == 0:
            avg_loss = running_loss / log_interval
            elapsed = time.time() - t_start
            tokens_done = (step + 1) * tokens_per_batch
            print(f'  Step {step+1}/{total_steps} | '
                  f'Loss: {avg_loss:.4f} | LR: {lr:.2e} | '
                  f'Tokens: {tokens_done/1e6:.0f}M | '
                  f'Time: {elapsed:.0f}s')
            running_loss = 0.0

        # Evaluation
        if (step + 1) % eval_every_steps == 0 or step == total_steps - 1:
            if model_type == 'mdm':
                val_loss = evaluate_mdm_loss(model, val_data_tensor, batch_size=batch_size)
            else:
                val_loss = evaluate_arm_loss(model, val_data_tensor, batch_size=batch_size, perm=perm)

            tokens_done = (step + 1) * tokens_per_batch
            training_log.append({
                'step': step + 1,
                'tokens': tokens_done,
                'val_loss': val_loss,
                'lr': lr
            })

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                if save_dir:
                    os.makedirs(save_dir, exist_ok=True)
                    torch.save(model.state_dict(),
                               os.path.join(save_dir, f'{name}_best.pt'))

            print(f'  ** Eval at step {step+1}: val_loss = {val_loss:.4f} '
                  f'(best = {best_val_loss:.4f})')

    elapsed_total = time.time() - t_start
    final_val_loss = training_log[-1]['val_loss'] if training_log else float('inf')

    print(f'\nDone training {name}: best_val={best_val_loss:.4f}, '
          f'final_val={final_val_loss:.4f}, time={elapsed_total:.0f}s')

    # Clean up GPU memory
    del model, optimizer
    torch.cuda.empty_cache()

    return {
        'best_val_loss': best_val_loss,
        'final_val_loss': final_val_loss,
        'n_params': n_params,
        'total_params': total_params,
        'training_tokens': total_training_tokens,
        'training_log': training_log,
        'elapsed_seconds': elapsed_total,
        'skipped': False
    }


print('Training pipeline defined.')

---
## Cell 11: Sample Fixed Permutations for π-Learners

We sample 3 fixed permutations from each of:
- uniform (fully random)
- closer (L/10 swaps from identity)
- much_closer (sqrt(L) swaps from identity)

Each π-learner trains with a single fixed permutation throughout.
Total: 9 π-learners + 1 ARM + 1 MDM = 11 model types.

In [ ]:
# Fix seed for reproducibility of permutation sampling
perm_rng = np.random.default_rng(seed=2025)

# Sample 3 permutations from each distribution
PI_LEARNERS = {}

for i in range(3):
    name = f'uniform_{i+1}'
    PI_LEARNERS[name] = sample_uniform_permutation(SEQ_LEN, rng=perm_rng)

for i in range(3):
    name = f'closer_{i+1}'
    PI_LEARNERS[name] = sample_closer_permutation(SEQ_LEN, rng=perm_rng)

for i in range(3):
    name = f'much_closer_{i+1}'
    PI_LEARNERS[name] = sample_much_closer_permutation(SEQ_LEN, rng=perm_rng)

# Save permutations for reproducibility
perm_save_path = f'{RESULTS_DIR}/permutations.npz'
np.savez(perm_save_path, **PI_LEARNERS)
print(f'Saved {len(PI_LEARNERS)} permutations to {perm_save_path}')

# Define all model types for the experiment
ALL_MODEL_TYPES = (
    [('arm', 'AR', None, None)] +
    [('mdm', 'MDM', None, None)] +
    [('pi_learner', f'pi_{name}', PI_LEARNERS[name], name)
     for name in PI_LEARNERS]
)

print(f'\nTotal model types: {len(ALL_MODEL_TYPES)}')
for mt, display, perm, pname in ALL_MODEL_TYPES:
    print(f'  {display} (type={mt}, perm={pname})')

---
## Cell 12: Run IsoFLOP Experiment

For each FLOPs budget C and each model type, we:
1. Try all candidate model sizes
2. Compute tokens = C / (6 * N_params)
3. Train and evaluate
4. Record the best (lowest val loss) across model sizes

Results are saved incrementally to Google Drive.

**WARNING**: This cell takes a long time to run. For 7 FLOPs budgets x 11 model types x ~3 model sizes each = ~230 training runs.

In [ ]:
# Convert validation data to tensor once
val_tensor = torch.from_numpy(val_data)

# Results storage
# Structure: results[display_name][flops_budget] = best_val_loss
results_path = f'{RESULTS_DIR}/isoflop_results.json'

# Try to load existing results (for resuming)
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        all_results = json.load(f)
    print(f'Loaded existing results from {results_path}')
else:
    all_results = {}
    print('Starting fresh experiment.')

# Detailed results for table
detailed_path = f'{RESULTS_DIR}/detailed_results.json'
if os.path.exists(detailed_path):
    with open(detailed_path, 'r') as f:
        detailed_results = json.load(f)
else:
    detailed_results = {}

total_runs = sum(
    len(FLOPS_TO_MODEL_SIZES[c]) for c in FLOPS_BUDGETS
) * len(ALL_MODEL_TYPES)
print(f'\nTotal training runs planned: {total_runs}')
print(f'This will take a while. Results are saved incrementally.\n')

run_counter = 0
experiment_start = time.time()

for model_type, display_name, perm, perm_name in ALL_MODEL_TYPES:
    if display_name not in all_results:
        all_results[display_name] = {}
    if display_name not in detailed_results:
        detailed_results[display_name] = {}

    for flops_budget in FLOPS_BUDGETS:
        flops_key = f'{flops_budget:.0e}'

        # Check if already computed
        if flops_key in all_results[display_name]:
            run_counter += len(FLOPS_TO_MODEL_SIZES[flops_budget])
            print(f'[SKIP] {display_name} @ C={flops_key}: '
                  f'already have result = {all_results[display_name][flops_key]:.4f}')
            continue

        best_loss_for_budget = float('inf')
        best_config_for_budget = None

        for model_size_name in FLOPS_TO_MODEL_SIZES[flops_budget]:
            run_counter += 1

            # Compute non-embedding params for this config
            cfg = MODEL_CONFIGS[model_size_name]
            d = cfg['d_model']
            L_layers = cfg['n_layers']
            n_params_approx = L_layers * (12 * d**2 + 13 * d) + d * VOCAB_SIZE

            # Chinchilla scaling: tokens = C / (6 * N)
            training_tokens = int(flops_budget / (6 * n_params_approx))

            # Cap at available data (can do multiple epochs)
            print(f'\n[Run {run_counter}/{total_runs}] '
                  f'{display_name} | C={flops_key} | Size={model_size_name} | '
                  f'N~{n_params_approx/1e6:.1f}M | Tokens={training_tokens/1e6:.0f}M')

            # Convert perm to torch tensor if needed
            perm_for_training = None
            if perm is not None:
                perm_for_training = torch.from_numpy(perm).long().to(device)

            result = train_single_model(
                model_type=model_type,
                model_config_name=model_size_name,
                total_training_tokens=training_tokens,
                train_data_np=train_data,
                val_data_tensor=val_tensor,
                perm=perm_for_training,
                perm_name=perm_name,
                batch_size=BATCH_SIZE,
                save_dir=MODELS_DIR
            )

            if result['best_val_loss'] < best_loss_for_budget:
                best_loss_for_budget = result['best_val_loss']
                best_config_for_budget = model_size_name

            # Save detailed result
            detail_key = f'{flops_key}_{model_size_name}'
            detailed_results[display_name][detail_key] = {
                'model_size': model_size_name,
                'flops': flops_budget,
                'n_params': result.get('n_params', n_params_approx),
                'training_tokens': training_tokens,
                'best_val_loss': result['best_val_loss'],
                'final_val_loss': result['final_val_loss'],
                'elapsed_seconds': result.get('elapsed_seconds', 0),
                'skipped': result.get('skipped', False)
            }

        # Record best for this FLOPs budget
        all_results[display_name][flops_key] = best_loss_for_budget
        print(f'\n*** {display_name} @ C={flops_key}: '
              f'best_val_loss = {best_loss_for_budget:.4f} '
              f'(config={best_config_for_budget}) ***')

        # Save incrementally
        with open(results_path, 'w') as f:
            json.dump(all_results, f, indent=2)
        with open(detailed_path, 'w') as f:
            json.dump(detailed_results, f, indent=2, default=str)

total_time = time.time() - experiment_start
print(f'\n{"="*60}')
print(f'IsoFLOP experiment complete!')
print(f'Total time: {total_time/3600:.1f} hours')
print(f'Results saved to: {results_path}')
print(f'Detailed results: {detailed_path}')

---
## Cell 13: Load Results & Prepare for Plotting

In [ ]:
# Load results (in case running from a fresh runtime after training)
results_path = f'{RESULTS_DIR}/isoflop_results.json'
with open(results_path, 'r') as f:
    all_results = json.load(f)

print('Loaded IsoFLOP results:')
for name, data in all_results.items():
    print(f'  {name}: {len(data)} FLOPs budgets')
    for fk, vl in sorted(data.items(), key=lambda x: float(x[0])):
        print(f'    C={fk}: val_loss={vl:.4f}')

---
## Cell 14: Plot Figure 2 (Left) — IsoFLOP Scaling Curves

x-axis: log(FLOPs), y-axis: -log p_θ(x) (validation loss)

Lines:
- AR (orange, thick)
- MDM (blue, thick)
- π-learner-much_closer (3 green lines)
- π-learner-closer (3 red/pink lines)
- π-learner-uniform (3 purple lines)

In [ ]:
# Publication-quality matplotlib settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 9,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'text.usetex': False,
    'mathtext.fontset': 'cm',
})


def extract_scaling_curve(results_dict, name):
    """Extract (flops, loss) pairs sorted by flops."""
    if name not in results_dict:
        return [], []
    items = [(float(k), v) for k, v in results_dict[name].items()]
    items.sort(key=lambda x: x[0])
    flops = [x[0] for x in items]
    losses = [x[1] for x in items]
    return flops, losses


fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Color scheme matching the paper
color_ar = '#E67E22'       # orange
color_mdm = '#2E86C1'      # blue
color_much_closer = '#27AE60'  # green
color_closer = '#E74C3C'    # red/pink
color_uniform = '#8E44AD'   # purple

# Plot AR
flops_ar, loss_ar = extract_scaling_curve(all_results, 'AR')
if flops_ar:
    ax.plot(flops_ar, loss_ar, 'o-', color=color_ar, linewidth=2.5,
            markersize=6, label='AR', zorder=10)

# Plot MDM
flops_mdm, loss_mdm = extract_scaling_curve(all_results, 'MDM')
if flops_mdm:
    ax.plot(flops_mdm, loss_mdm, 's-', color=color_mdm, linewidth=2.5,
            markersize=6, label='MDM', zorder=10)

# Plot pi-learners: much_closer (3 lines)
for i in range(1, 4):
    name = f'pi_much_closer_{i}'
    flops_i, loss_i = extract_scaling_curve(all_results, name)
    if flops_i:
        label = r'$\pi$-learner (much closer)' if i == 1 else None
        ax.plot(flops_i, loss_i, '^--', color=color_much_closer,
                linewidth=1.2, markersize=4, alpha=0.8, label=label)

# Plot pi-learners: closer (3 lines)
for i in range(1, 4):
    name = f'pi_closer_{i}'
    flops_i, loss_i = extract_scaling_curve(all_results, name)
    if flops_i:
        label = r'$\pi$-learner (closer)' if i == 1 else None
        ax.plot(flops_i, loss_i, 'v--', color=color_closer,
                linewidth=1.2, markersize=4, alpha=0.8, label=label)

# Plot pi-learners: uniform (3 lines)
for i in range(1, 4):
    name = f'pi_uniform_{i}'
    flops_i, loss_i = extract_scaling_curve(all_results, name)
    if flops_i:
        label = r'$\pi$-learner (uniform)' if i == 1 else None
        ax.plot(flops_i, loss_i, 'D--', color=color_uniform,
                linewidth=1.2, markersize=4, alpha=0.8, label=label)

ax.set_xscale('log')
ax.set_xlabel('FLOPs')
ax.set_ylabel(r'$-\log\, p_\theta(x)$ (validation loss)')
ax.set_title('IsoFLOP Scaling Laws: ARM vs MDM vs $\\pi$-Learners\n(SlimPajama, L=2048)')
ax.legend(loc='upper right', framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

# Save
fig2_left_path = f'{FIGURES_DIR}/figure2_left_isoflop_scaling.pdf'
fig2_left_png = f'{FIGURES_DIR}/figure2_left_isoflop_scaling.png'
fig.savefig(fig2_left_path)
fig.savefig(fig2_left_png)
print(f'Saved Figure 2 (Left) to:')
print(f'  {fig2_left_path}')
print(f'  {fig2_left_png}')
plt.show()

---
## Cell 15: Figure 2 (Top-Right) — Validation Loss Table

Table comparing validation loss for:
- π = id (ARM)
- π ~ Closer(S_L) (average of 3 closer π-learners)
- π ~ Unif(S_L) (average of 3 uniform π-learners)

At the largest FLOPs budget.

In [ ]:
# Find the largest FLOPs budget with results
available_flops = set()
for name, data in all_results.items():
    for fk in data.keys():
        available_flops.add(float(fk))
available_flops = sorted(available_flops)
print(f'Available FLOPs budgets: {[f"{x:.0e}" for x in available_flops]}')

# Build table data for each FLOPs budget
table_data = []

for c in available_flops:
    flops_key = f'{c:.0e}'
    row = {'FLOPs': flops_key}

    # AR (pi = id)
    if 'AR' in all_results and flops_key in all_results['AR']:
        row['pi=id (AR)'] = all_results['AR'][flops_key]
    else:
        row['pi=id (AR)'] = None

    # MDM
    if 'MDM' in all_results and flops_key in all_results['MDM']:
        row['MDM'] = all_results['MDM'][flops_key]
    else:
        row['MDM'] = None

    # Closer: average of 3 pi-learners
    closer_losses = []
    for i in range(1, 4):
        name = f'pi_closer_{i}'
        if name in all_results and flops_key in all_results[name]:
            closer_losses.append(all_results[name][flops_key])
    if closer_losses:
        row['pi~Closer'] = np.mean(closer_losses)
        row['pi~Closer_std'] = np.std(closer_losses)
    else:
        row['pi~Closer'] = None
        row['pi~Closer_std'] = None

    # Much closer: average of 3 pi-learners
    mc_losses = []
    for i in range(1, 4):
        name = f'pi_much_closer_{i}'
        if name in all_results and flops_key in all_results[name]:
            mc_losses.append(all_results[name][flops_key])
    if mc_losses:
        row['pi~MuchCloser'] = np.mean(mc_losses)
        row['pi~MuchCloser_std'] = np.std(mc_losses)
    else:
        row['pi~MuchCloser'] = None
        row['pi~MuchCloser_std'] = None

    # Uniform: average of 3 pi-learners
    unif_losses = []
    for i in range(1, 4):
        name = f'pi_uniform_{i}'
        if name in all_results and flops_key in all_results[name]:
            unif_losses.append(all_results[name][flops_key])
    if unif_losses:
        row['pi~Unif'] = np.mean(unif_losses)
        row['pi~Unif_std'] = np.std(unif_losses)
    else:
        row['pi~Unif'] = None
        row['pi~Unif_std'] = None

    table_data.append(row)

# Print table
print('\n' + '='*90)
print('Figure 2 (Top-Right): Validation Loss Comparison')
print('='*90)
header = f'{"FLOPs":>10s} | {"pi=id (AR)":>12s} | {"MDM":>12s} | {"pi~MuchCloser":>16s} | {"pi~Closer":>16s} | {"pi~Unif":>16s}'
print(header)
print('-'*90)

for row in table_data:
    parts = [f'{row["FLOPs"]:>10s}']

    for key in ['pi=id (AR)', 'MDM']:
        if row[key] is not None:
            parts.append(f'{row[key]:>12.4f}')
        else:
            parts.append(f'{"--":>12s}')

    for key, std_key in [('pi~MuchCloser', 'pi~MuchCloser_std'),
                          ('pi~Closer', 'pi~Closer_std'),
                          ('pi~Unif', 'pi~Unif_std')]:
        if row[key] is not None:
            parts.append(f'{row[key]:>10.4f}+/-{row[std_key]:.4f}')
        else:
            parts.append(f'{"--":>16s}')

    print(' | '.join(parts))

print('='*90)

In [ ]:
# Render the table as a matplotlib figure (publication quality)

fig_table, ax_table = plt.subplots(figsize=(12, max(3, 1 + 0.5 * len(table_data))))
ax_table.axis('off')

# Prepare cell text
col_labels = ['FLOPs', r'$\pi$=id (AR)', 'MDM',
              r'$\pi{\sim}$MuchCloser', r'$\pi{\sim}$Closer', r'$\pi{\sim}$Unif']

cell_text = []
cell_colors = []
for row in table_data:
    crow = [row['FLOPs']]
    ccol = ['#f7f7f7']

    # Collect all valid losses to find best
    losses_in_row = []
    for key in ['pi=id (AR)', 'MDM', 'pi~MuchCloser', 'pi~Closer', 'pi~Unif']:
        if row.get(key) is not None:
            losses_in_row.append(row[key])
    best_in_row = min(losses_in_row) if losses_in_row else None

    for key, std_key in [('pi=id (AR)', None), ('MDM', None),
                          ('pi~MuchCloser', 'pi~MuchCloser_std'),
                          ('pi~Closer', 'pi~Closer_std'),
                          ('pi~Unif', 'pi~Unif_std')]:
        if row.get(key) is not None:
            if std_key and row.get(std_key) is not None:
                crow.append(f'{row[key]:.3f} +/- {row[std_key]:.3f}')
            else:
                crow.append(f'{row[key]:.3f}')
            # Highlight best
            if best_in_row and abs(row[key] - best_in_row) < 1e-6:
                ccol.append('#d4efdf')  # light green
            else:
                ccol.append('white')
        else:
            crow.append('--')
            ccol.append('white')

    cell_text.append(crow)
    cell_colors.append(ccol)

table = ax_table.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellColours=cell_colors,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.5)

# Style header row
for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor('#2c3e50')
    cell.set_text_props(color='white', fontweight='bold')

ax_table.set_title(
    'Figure 2 (Top-Right): Validation Loss by Permutation Distribution\n'
    '(SlimPajama, L=2048, best highlighted)',
    fontsize=13, fontweight='bold', pad=20
)

plt.tight_layout()

fig2_right_path = f'{FIGURES_DIR}/figure2_topright_table.pdf'
fig2_right_png = f'{FIGURES_DIR}/figure2_topright_table.png'
fig_table.savefig(fig2_right_path)
fig_table.savefig(fig2_right_png)
print(f'Saved Figure 2 (Top-Right) to:')
print(f'  {fig2_right_path}')
print(f'  {fig2_right_png}')
plt.show()

---
## Cell 16: Save All Results Summary

In [ ]:
# Final summary
summary = {
    'experiment': 'IsoFLOP Scaling Laws (Figure 2)',
    'paper': 'Train for the Worst, Plan for the Best (Kim et al., ICML 2025)',
    'arxiv': '2502.06768v3',
    'dataset': 'SlimPajama',
    'seq_len': SEQ_LEN,
    'vocab_size': VOCAB_SIZE,
    'tokenizer': 'GPT-2 + mask token at index 0',
    'positional_embedding': 'learned (NOT RoPE)',
    'model_configs': MODEL_CONFIGS,
    'flops_budgets': FLOPS_BUDGETS,
    'optimizer': 'AdamW (beta1=0.9, beta2=0.95, wd=0.1)',
    'lr_schedule': 'cosine, max_lr=4e-4, min_lr=4e-5',
    'batch_size': BATCH_SIZE,
    'permutation_distributions': {
        'identity': 'pi=id (standard AR)',
        'much_closer': f'sqrt(L)={int(math.sqrt(SEQ_LEN))} random transposition swaps from id',
        'closer': f'L/10={SEQ_LEN//10} random transposition swaps from id',
        'uniform': 'fully random permutation'
    },
    'n_pi_samples_per_dist': 3,
    'isoflop_results': all_results,
}

summary_path = f'{RESULTS_DIR}/experiment_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Experiment summary saved to: {summary_path}')
print(f'\nAll output files:')
print(f'  Results: {results_path}')
print(f'  Detailed: {detailed_path}')
print(f'  Summary: {summary_path}')
print(f'  Figure 2 (Left): {fig2_left_path}')
print(f'  Figure 2 (Top-Right): {fig2_right_path}')
print(f'  Permutations: {perm_save_path}')
print(f'\nDone!')